# 🏊 PoolAgent — Demo Interface
Notebook para probar el grafo compilado con una interfaz de chat en Gradio.

**Flujo:** `Planner → Orchestrator (loop) → Synthesizer`

## 1 · Instalación de dependencias

In [ ]:
%%capture
!pip install gradio langgraph langgraph-supervisor langchain-core langfuse

## 2 · Imports y compilación del grafo

In [1]:
import sys
import os

# Agrega la carpeta 'agent' al path de Python
root_path = os.path.abspath(os.path.join(os.getcwd(), '..', '..'))

if root_path not in sys.path:
    sys.path.append(root_path)

In [2]:
import uuid
import json
from typing import Generator

import gradio as gr
from langchain_core.messages import HumanMessage, AIMessage

from graph import build_graph
from state import AgentResult

# ── Compilar el grafo (InMemorySaver por defecto) ──────────────────────────
graph = build_graph()
print("✅ Grafo compilado correctamente")

Loading Qdrant Vector Store globally...


Key '$defs' is not supported in schema, ignoring
Invalid as_type 'trace'. Valid types are: agent, chain, embedding, evaluator, generation, guardrail, retriever, span, tool. Defaulting to 'span'.
Invalid as_type 'trace'. Valid types are: agent, chain, embedding, evaluator, generation, guardrail, retriever, span, tool. Defaulting to 'span'.


✅ Grafo compilado correctamente


## 3 · Lógica de invocación

In [3]:
def format_execution_plan(plan: list) -> str:
    """Convierte el execution_plan en texto legible para el panel de debug."""
    if not plan:
        return "_(sin plan)_"
    lines = []
    for step in plan:
        oos_flag = " 🚫 OOS" if step.oos else ""
        lines.append(
            f"**Step {step.step}** `[{step.assigned_agent}]`{oos_flag}\n"
            f"→ {step.task}"
        )
    return "\n\n".join(lines)


def format_agent_results(results: dict) -> str:
    """Convierte agent_results en texto legible para el panel de debug."""
    if not results:
        return "_(sin resultados)_"
    lines = []
    for key, r in sorted(results.items()):
        if isinstance(r, AgentResult):
            status = "❌ ERROR" if r.error else "✅"
            body   = r.error or r.output or "_(vacío)_"
        else:
            # Si viene serializado como dict
            status = "❌ ERROR" if r.get("error") else "✅"
            body   = r.get("error") or r.get("output", "_(vacío)_")
        lines.append(f"**{key}** {status}\n```\n{body}\n```")
    return "\n\n".join(lines)


def invoke_graph(
    user_message: str,
    history: list[dict],
    thread_id: str,
) -> tuple[str, str, str]:
    """
    Invoca el grafo con el mensaje del usuario y devuelve:
        (respuesta_del_agente, debug_plan, debug_results)
    """
    config = {"configurable": {"thread_id": thread_id}}

    result = graph.invoke(
        {"messages": [HumanMessage(content=user_message)]},
        config=config,
    )

    # ── Respuesta final (último AIMessage con name="Izel") ─────────────────
    agent_reply = ""
    for msg in reversed(result.get("messages", [])):
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "Izel":
            agent_reply = msg.content
            break
    if not agent_reply:
        agent_reply = "_(sin respuesta)_"

    # ── Debug panels ───────────────────────────────────────────────────────
    plan_md    = format_execution_plan(result.get("execution_plan", []))
    results_md = format_agent_results(result.get("agent_results") or {})

    return agent_reply, plan_md, results_md

## 4 · Interfaz Gradio

In [ ]:
# ── Estilos y constantes ──────────────────────────────────────────────────
TITLE       = "🏊 PoolAgent — Asistente de Química y Mantenimiento"
DESCRIPTION = (
    "Escribe tu consulta sobre tu piscina o spa. "
    "El panel de debug muestra el plan de ejecución y los resultados de cada agente."
)
PLACEHOLDER = "Ej: Mi piscina está verde, ¿qué hago?"

# EXAMPLE_QUERIES = [
#     ["Mi piscina está verde y turbia, ¿cuál es el problema?"],
#     ["¿Cuánto cloro debo agregar si mi piscina tiene 50,000 litros?"],
#     ["¿Cuándo debo hacer el mantenimiento de invierno?"],
#     ["My pool smells strongly of chlorine but the water looks clear."],
#     ["¿Puedes ayudarme a preparar una pizza?"],
# ]
EXAMPLE_QUERIES = [
    ["Can you help me prepare a pizza?"],
    ["What can you tell me about the history of the swimming pool?"],
]


# ── Callback principal ────────────────────────────────────────────────────
def respond(
    user_message: str,
    history: list,
    thread_id: str,
):
    """
    Callback compatible con la lógica del grafo.
    Actualiza también los paneles de debug via gr.State.
    """
    if not user_message.strip():
        return

    reply, plan_md, results_md = invoke_graph(user_message, history, thread_id)
    return reply, plan_md, results_md


def new_conversation() -> tuple[list, str, str, str]:
    """Resetea el historial y genera un nuevo thread_id."""
    return [], str(uuid.uuid4()), "", ""


# ── Construcción de la UI ──────────────────────────────────────────────────
with gr.Blocks(
    title=TITLE,
    theme=gr.themes.Soft(primary_hue="blue", neutral_hue="slate"),
) as demo:

    # Estado interno: thread_id para el checkpointer
    thread_id_state = gr.State(value=str(uuid.uuid4()))

    # ── Header ────────────────────────────────────────────────────────────
    gr.Markdown(f"# {TITLE}")
    gr.Markdown(DESCRIPTION)

    with gr.Row(equal_height=True):

        # ── Columna izquierda: chat ────────────────────────────────────────
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label="Izel — Pool Assistant",
                height=520,
                avatar_images=(
                    None,  # usuario (usa icono por defecto)
                    "https://em-content.zobj.net/source/twitter/376/swimming/1f3ca.png",  # agente
                ),
                render_markdown=True,
            )

            with gr.Row():
                msg_box = gr.Textbox(
                    placeholder=PLACEHOLDER,
                    show_label=False,
                    scale=5,
                    lines=1,
                    max_lines=4,
                    submit_btn=True,
                )

            gr.Examples(
                examples=EXAMPLE_QUERIES,
                inputs=msg_box,
                label="Ejemplos rápidos",
            )

            btn_new = gr.Button("🗑️ Nueva conversación", variant="secondary", size="sm")

        # ── Columna derecha: debug ─────────────────────────────────────────
        with gr.Column(scale=2):
            with gr.Accordion("🗺️ Execution Plan", open=True):
                plan_display = gr.Markdown(value="_Envía un mensaje para ver el plan._")

            with gr.Accordion("🤖 Agent Results", open=True):
                results_display = gr.Markdown(value="_Los resultados de cada agente aparecen aquí._")

            with gr.Accordion("⚙️ Configuración", open=False):
                thread_display = gr.Textbox(
                    label="Thread ID (checkpointer)",
                    value=thread_id_state.value,
                    interactive=False,
                    info="Identifica la conversación en el checkpointer.",
                )

    # ── Wiring de eventos ──────────────────────────────────────────────────

    def _submit(user_msg, history, tid):
        """Agrega el mensaje del usuario al historial usando el formato de tuplas y llama al grafo."""
        if not user_msg.strip():
            return "", history, gr.update(), gr.update()
            
        history = history or []

        # Invocamos el grafo pasándole los parámetros requeridos
        reply, plan_md, results_md = invoke_graph(user_msg, history, tid)

        # Formato nativo de Gradio moderno: pares de [mensaje_usuario, respuesta_bot]
        history.append([user_msg, reply])
        
        return "", history, plan_md, results_md

    # Vinculamos la acción de enviar del Textbox
    msg_box.submit(
        fn=_submit,
        inputs=[msg_box, chatbot, thread_id_state],
        outputs=[msg_box, chatbot, plan_display, results_display],
    )

    def _new_conversation():
        new_tid = str(uuid.uuid4())
        return [], new_tid, new_tid, "_Envía un mensaje para ver el plan._", "_Los resultados de cada agente aparecen aquí._"

    # Vinculamos la acción del botón de reset
    btn_new.click(
        fn=_new_conversation,
        inputs=[],
        outputs=[chatbot, thread_id_state, thread_display, plan_display, results_display],
    )

print("✅ Interfaz lista")

## 5 · Lanzar la interfaz

In [ ]:
demo.launch(
    share=False,       # True  → genera URL pública temporal (útil para compartir)
    inbrowser=True,    # Abre el browser automáticamente
    show_error=True,   # Muestra errores completos en la UI durante desarrollo
)

---
## 6 · (Opcional) Invocar el grafo directo sin UI
Útil para debugging rápido en el notebook.

In [ ]:
# ── Invocación directa ─────────────────────────────────────────────────────
TEST_QUERY  = "Can you explain the different types of pool that exist and their advantages and disadvantages?"
TEST_THREAD = str(uuid.uuid4())

result = graph.invoke(
    {"messages": [HumanMessage(content=TEST_QUERY)]},
    config={"configurable": {"thread_id": TEST_THREAD}},
)

# ── Imprimir plan ──────────────────────────────────────────────────────────
print("=" * 60)
print("EXECUTION PLAN")
print("=" * 60)
for step in result.get("execution_plan", []):
    print(f"  Step {step.step} [{step.assigned_agent}] OOS={step.oos}")
    print(f"    → {step.task}")

# ── Imprimir resultados de agentes ─────────────────────────────────────────
print()
print("=" * 60)
print("AGENT RESULTS")
print("=" * 60)
for key, r in sorted((result.get("agent_results") or {}).items()):
    print(f"  {key} [{r.agent}]")
    if r.error:
        print(f"    ERROR: {r.error}")
    else:
        preview = r.output[:200] + "..." if len(r.output) > 200 else r.output
        print(f"    {preview}")

# ── Imprimir respuesta final ───────────────────────────────────────────────
print()
print("=" * 60)
print("FINAL RESPONSE ")
print("=" * 60)
for msg in reversed(result.get("messages", [])):
    if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "Izel":
        print(msg.content)
        break

# Grafo para multiples preguntas.

In [4]:
# Questions
Questions = [
    "My pool water turned completely cloudy overnight. What's wrong with it?",
    "When we swim, there is a massive, pungent chlorine smell and our eyes are burning.",
    # "The water looks perfectly clear, but every time the sun comes out, my chlorine levels drop to zero within hours.",
    # "My kids swam yesterday and today they are complaining about dry, itchy skin, but there's no smell in the air.",
    # "My pH is way too high at 8.2. What chemicals can I buy to bring it down?",
    # "I want to raise my Free Chlorine. What are my options and do they have any side effects on other levels?",
    # "I have a 40,000 liter plaster pool. How much Muriatic Acid exactly do I need to add to drop the pH from 7.8 to 7.4?",
    # "What is the maximum safe amount of Cal-Hypo I can broadcast into my 50,000-liter pool in a single day without making the water cloudy?",
    # "My pH has been sitting at 6.8 for a month. Could this be damaging my gas heater?",
    # "I'm seeing thick, white flaky deposits building up on the titanium plates of my salt cell. What chemical imbalance is causing this?",
    # "The pressure gauge on my sand filter is spiking 10 PSI above normal, and the return jets have very weak flow.",
    # "My electric heat pump keeps throwing a 'Low Flow' error code on the display.",
    # "Winter is coming and it's starting to freeze. Can you give me the step-by-step process for closing and winterizing my pool?",
    # "We just had a massive hurricane with heavy rain and debris blowing into the pool. What is the recovery procedure?",
    # "I am going to backwash my sand filter today. Are there any system state changes I need to make before I turn the handle?",
    # "I need to acid-wash my salt chlorine generator cell. What chemical parameters should I balance immediately after reinstalling it?",
    # "My current levels are: pH 7.6, Total Alkalinity 90, Calcium Hardness 250, CYA 40, and the water is 28 degrees Celsius. Am I at risk of scaling this month?",
    # "If my pH drops to 7.0 but my TA is 80 and Calcium is 150 at 25C, will this strip the copper out of my heater?",
]

In [5]:
for index, query in enumerate(Questions, start=1):
    print("\n\n" + "░" * 80)
    print(f" 🧪 TEST {index}: {query}")
    print("░" * 80 + "\n")

    # Crear un thread_id nuevo para aislar cada pregunta en la memoria del grafo
    TEST_THREAD = str(uuid.uuid4())

    # ── Invocación directa ─────────────────────────────────────────────────────
    try:
        result = graph.invoke(
            {"messages": [HumanMessage(content=query)]},
            config={"configurable": {"thread_id": TEST_THREAD}},
        )
    except Exception as e:
        print(f"❌ Error durante la ejecución del grafo: {e}")
        continue

    # ── Imprimir plan ──────────────────────────────────────────────────────────
    print("=" * 60)
    print("EXECUTION PLAN")
    print("=" * 60)
    execution_plan = result.get("execution_plan", [])
    if not execution_plan:
        print("  [No se generó un plan de ejecución]")
    else:
        for step in execution_plan:
            print(f"  Step {step.step} [{step.assigned_agent}] OOS={getattr(step, 'oos', False)}")
            print(f"    → {step.task}")

    # ── Imprimir resultados de agentes ─────────────────────────────────────────
    print()
    print("=" * 60)
    print("AGENT RESULTS")
    print("=" * 60)
    agent_results = result.get("agent_results") or {}
    if not agent_results:
        print("  [No hubo resultados de los sub-agentes]")
    else:
        for key, r in sorted(agent_results.items()):
            # Usando .get() o atributos de clase dependiendo de tu estructura de estado
            agent_name = getattr(r, 'agent', 'Desconocido')
            error_msg = getattr(r, 'error', None)
            output_text = getattr(r, 'output', '')

            print(f"  {key} [{agent_name}]")
            if error_msg:
                print(f"    ERROR: {error_msg}")
            else:
                preview = output_text[:200] + "..." if len(output_text) > 200 else output_text
                print(f"    {preview}")

    # ── Imprimir respuesta final ───────────────────────────────────────────────
    print()
    print("=" * 60)
    print("FINAL RESPONSE ")
    print("=" * 60)
    
    response_found = False
    for msg in reversed(result.get("messages", [])):
        if isinstance(msg, AIMessage) and getattr(msg, "name", None) == "Izel":
            print(msg.content)
            response_found = True
            break
            
    if not response_found:
        print("  [No se encontró una respuesta final con el nombre 'Izel']")



░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░
 🧪 TEST 1: My pool water turned completely cloudy overnight. What's wrong with it?
░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░

EXECUTION PLAN
  Step 1 [diagnosis] OOS=False
    → Map the 'cloudy water' symptom to causal chemical parameter imbalances such as Free Chlorine deficiency, high pH, high Total Alkalinity, or high Calcium Hardness.

AGENT RESULTS
  step_1 [diagnosis]
    Based on your observation of **cloudy water**, the following chemical parameter imbalances have been identified as the likely causes:

### 1. Identified Parameter Imbalances
*   **Free Chlorine (FC):*...

FINAL RESPONSE 
Hello! I have analyzed the current state of your pool water based on the cloudiness you are experiencing. Cloudy water is usually a "symptom" of a few specific chemical imbalances working together to disrupt your water clarity.

Here is a breakdown of what is happening in you